In [ ]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
from tqdm import tqdm

from sqlalchemy import create_engine
from urllib.parse import quote_plus

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import utils.prediction_utils as pred
from utils import dictionaries as dictionary
from utils.logger import setup_logger

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Suppress all warnings
warnings.filterwarnings('ignore')

: 

In [1]:
class ModelTrainer:
    def __init__(self, test_set_date, cut_off_date, target_groups, numerical_cols, categorical_cols, boolean_cols, target, model_params):
        self.test_set_date = test_set_date
        self.cut_off_date = cut_off_date
        self.target_groups = target_groups
        self.numerical_cols = numerical_cols
        self.categorical_cols = categorical_cols
        self.boolean_cols = boolean_cols
        self.features = numerical_cols + categorical_cols + boolean_cols
        self.target = target
        self.model_params = model_params

        self.query = self.construct_query()
        self.df = self.fetch_and_prep_data(self.query)
        if self.df is None:
            raise ValueError("DataFrame could not be fetched and prepared.")
        self.train_data, self.test_data = self.split_data(self.df, self.test_set_date)
        self.X_train, self.y_train, self.X_test, self.y_test = self.prepare_features_and_target(self.train_data, self.test_data)

        self.best_model = None
        self.best_params = None
        self.results = []
        self.models_df = pd.DataFrame()
        self.shap_values = None
        self.grid_search_objects = []

        self.preprocessor_unscaled = self.create_preprocessor(scaling=False)
        self.preprocessor_scaled = self.create_preprocessor(scaling=True)
        self.models = self.create_models()

    def construct_query(self):
        formatted_target_groups = "', '".join(self.target_groups)
        formatted_target_groups = f"('{formatted_target_groups}')"

        query = f'''
            SELECT b.flight_id,
                   b.target_group,
                   f.advertiser,
                   f.product,
                   f.start_date,
                   f.end_date,
                   b.campaign_duration,
                   b.daypart,
                   f.mock_campaign,
                   f.alcohol_kansspelen,
                   b.total_grp,
                   b.ster_grp,
                   b.regio_grp,
                   b.npo1_grp,
                   b.npo2_grp,
                   b.npo3_grp,
                   b.[ster_1+]
            FROM [dbo].[tv.bereik_ster] b
            LEFT JOIN [dbo].[tv.flights_ster] f ON b.flight_id = f.flight_id
            WHERE b.ster_grp > 0 AND target_group IN {formatted_target_groups}
              AND start_date >= '{self.cut_off_date}'
            ORDER BY start_date
        '''
        return query

    def create_sql_alchemy_engine(self):
        """
        Creates and returns a SQLAlchemy engine for connecting to a SQL Server database.
        """
        quoted = quote_plus(
            "Driver={ODBC+Driver+17+for+SQL+Server};"
            "Server=tcp:stercampsql.database.windows.net;"
            "Database=SterCampNew;"
            "uid=stercampadmin;"
            f"pwd={os.getenv('SQLDB')}"
        )
        alchemy_con = f"mssql+pyodbc:///?odbc_connect={quoted}"
        engine = create_engine(alchemy_con)
        return engine

    def query_to_df(self, query):
        engine = self.create_sql_alchemy_engine()
        df_query = pd.read_sql_query(query, engine)
        return df_query

    def fetch_and_prep_data(self, query):
        try:
            df = self.query_to_df(query)
            df = self.prep_dataframe(df)
        except Exception as e:
            print(f"Error executing query: {str(e)}")
            return None
        return df

    def prep_dataframe(self, df_bereik):


        df = df_bereik[["flight_id", "target_group", "start_date", "end_date", "advertiser", "product", "daypart", "mock_campaign", "alcohol_kansspelen",
                        "npo1_grp", "npo2_grp", "npo3_grp", "regio_grp", "ster_grp",
                        "ster_1+"]].reset_index(drop=True)

        df = (df.pivot_table(index=["flight_id", "target_group", "start_date", "end_date", 'advertiser', 'product', "mock_campaign", "alcohol_kansspelen"],
                            columns='daypart', fill_value=0.0,
                            values=['npo1_grp', 'npo2_grp', 'npo3_grp',
                                    'ster_grp', 'ster_1+', 'regio_grp'], aggfunc='sum').reset_index())

        df.columns = ["_".join(col) if col[1] else col[0] for col in df.columns.to_flat_index()]

        # Convert start_date and end_date to datetime objects and calculate required values
        df['start_date'] = pd.to_datetime(df['start_date'], format="%d%m%y", errors='coerce')
        df['end_date'] = pd.to_datetime(df['end_date'], format="%d%m%y", errors='coerce')

        # Apply function to create extra columns
        df['timef_start_month'] = df['start_date'].dt.month
        df['timef_end_month'] = df['end_date'].dt.month
        df['timef_start_year'] = df['start_date'].dt.year
        df['timef_end_year'] = df['end_date'].dt.year
        df['timef_start_week'] = df['start_date'].dt.strftime("%V").astype(int)
        df['timef_end_week'] = df['end_date'].dt.strftime("%V").astype(int)
        df['timef_start_day'] = df['start_date'].dt.day
        df['timef_end_day'] = df['end_date'].dt.day
        df['timef_duration_days'] = df.apply(lambda row: (row['end_date'] - row['start_date']).days + 1, axis=1)

        df['timef_start_season'] = df['timef_start_month'].apply(get_season)
        df['timef_end_season'] = df['timef_end_month'].apply(get_season)

        # Calculate total overlap with olympics and WK/EK
        df['timef_olympic_period_overlap'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.olympic_period_list))
        df['timef_football_period_overlap'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.football_wk_ek_list))

        df = df.sort_values(by='start_date')
        # Create a new column 'data_type' indicating old or new
        df['data_type'] = pd.cut(df['start_date'], bins=[pd.Timestamp.min, pd.to_datetime('2023-09-01'), pd.Timestamp.max], labels=[0, 1])

        return df

    def split_data(self, df, test_set_date):
        train_data = df[df['start_date'] < test_set_date]
        test_data = df[df['start_date'] >= test_set_date]
        return train_data, test_data

    def prepare_features_and_target(self, train_data, test_data):
        X_train = train_data[self.features]
        y_train = train_data[self.target]
        X_test = test_data[self.features]
        y_test = test_data[self.target]
        return X_train, y_train, X_test, y_test

    def create_preprocessor(self, scaling=True):
        numerical_transformer_steps = [('imputer', SimpleImputer(strategy='mean'))]
        if scaling:
            numerical_transformer_steps.append(('scaler', StandardScaler()))

        numerical_transformer = Pipeline(steps=numerical_transformer_steps)

        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))])

        boolean_transformer = Pipeline(steps=[
            ('int_conversion', FunctionTransformer(lambda x: x.astype(int), validate=False)),
            ('imputer', SimpleImputer(strategy='most_frequent'))])

        transformers = []
        transformers.append(('num', numerical_transformer, self.numerical_cols))
        transformers.append(('cat', categorical_transformer, self.categorical_cols))
        transformers.append(('bool', boolean_transformer, self.boolean_cols))

        preprocessor = ColumnTransformer(transformers=transformers)

        return preprocessor

    def create_models(self):
        models = []
        for model_info in self.model_params:
            models.append({
                'name': model_info['name'],
                'model': model_info['model'],
                'params': model_info['params'],
                'preprocessor': self.preprocessor_unscaled if model_info['scaling'] == 'unscaled' else self.preprocessor_scaled
            })
        return models

    def grid_search_models(self):
        models_data = {
            'Model': [],
            'Parameters': [],
            'R2 Train': [],
            'R2 Test': [],
            'MSE Train': [],
            'MSE Test': [],
            'MAE Train': [],
            'MAE Test': []
        }

        tscv = TimeSeriesSplit(n_splits=5)
        total_iterations = sum(len(ParameterGrid(model_info['params'])) for model_info in self.models) * tscv.get_n_splits(self.X_train)
        pbar = tqdm(total=total_iterations, desc="Grid Search Progress")

        for model_info in self.models:
            pipeline = Pipeline(steps=[
                ('preprocessor', model_info['preprocessor']),
                ('model', model_info['model'])
            ])

            grid_search = GridSearchCV(
                estimator=pipeline,
                param_grid=model_info['params'],
                cv=tscv,
                scoring='neg_mean_squared_error',
                n_jobs=-1,
                return_train_score=True,
                verbose=0
            )

            grid_search.fit(self.X_train, self.y_train)
            pbar.update(len(ParameterGrid(model_info['params'])) * tscv.get_n_splits(self.X_train))

            self.grid_search_objects.append(grid_search)
            results_df = pd.DataFrame(grid_search.cv_results_)
            self.results.append(results_df)

            if grid_search.best_score_ > (self.best_score if self.best_model else float('-inf')):
                self.best_score = grid_search.best_score_
                self.best_model = grid_search.best_estimator_
                self.best_params = grid_search.best_params_

            y_train_pred = grid_search.predict(self.X_train)
            y_test_pred = grid_search.predict(self.X_test)

            r2_train = r2_score(self.y_train, y_train_pred)
            r2_test = r2_score(self.y_test, y_test_pred)
            mse_train = mean_squared_error(self.y_train, y_train_pred)
            mse_test = mean_squared_error(self.y_test, y_test_pred)
            mae_train = mean_absolute_error(self.y_train, y_train_pred)
            mae_test = mean_absolute_error(self.y_test, y_test_pred)

            models_data['Model'].append(model_info['name'])
            models_data['Parameters'].append(grid_search.best_params_)
            models_data['R2 Train'].append(r2_train)
            models_data['R2 Test'].append(r2_test)
            models_data['MSE Train'].append(mse_train)
            models_data['MSE Test'].append(mse_test)
            models_data['MAE Train'].append(mae_train)
            models_data['MAE Test'].append(mae_test)

        pbar.close()
        self.models_df = pd.DataFrame(models_data)
        print(f"Best Model: {self.best_model}")
        print(f"Best Parameters: {self.best_params}")

    def extract_shap_values(self):
        if not isinstance(self.best_model.named_steps['model'], (XGBRegressor, LGBMRegressor)):
            print("The best model is not a tree-based model. SHAP values cannot be extracted.")
            return

        X_test_transformed = self.best_model.named_steps['preprocessor'].transform(self.X_test)
        explainer = shap.Explainer(self.best_model.named_steps['model'])
        self.shap_values = explainer(X_test_transformed)
        print("SHAP values extracted.")

    def visualize_shap_values(self):
        if self.shap_values is None:
            print("No SHAP values to visualize.")
            return

        shap.summary_plot(self.shap_values, self.X_test)
        shap.dependence_plot(0, self.shap_values, self.X_test)
        shap.summary_plot(self.shap_values, self.X_test, plot_type="bar")

In [ ]:
TEST_SET_DATE = pd.to_datetime('2024-05-01')
CUT_OFF_DATE = '2023-09-01'
TARGET_GROUPS = ['25-67']

NUMERICAL_COLS = ['npo1_grp_daynight', 'npo1_grp_primetime', 'npo1_grp_total',
                  'npo2_grp_daynight', 'npo2_grp_primetime', 'npo2_grp_total',
                  'npo3_grp_daynight', 'npo3_grp_primetime', 'npo3_grp_total',
                  'regio_grp_daynight', 'regio_grp_primetime', 'regio_grp_total',
                  'timef_start_month', 'timef_end_month', 'timef_start_week', 'timef_end_week',
                  'timef_start_year', 'timef_end_year', 'timef_start_day', 'timef_end_day',
                  'timef_duration_days', 'timef_olympic_period_overlap', 'timef_football_period_overlap']

CATEGORICAL_COLS = ['daypart', 'mock_campaign', 'alcohol_kansspelen']
BOOLEAN_COLS = ['data_type']
TARGET = 'total_grp'

MODEL_PARAMS = [
    {
        'name': 'XGB without scaling',
        'model': XGBRegressor(objective='reg:squarederror'),
        'params': {
            'model__colsample_bytree': [0.8, 1.0],
            'model__gamma': [0.4, 0.6, 0.8],
            'model__learning_rate': [0.01, 0.05, 0.1],
            'model__max_depth': [5, 10, 15],
            'model__min_child_weight': [1, 5, 10],
            'model__n_estimators': [200, 400, 600],
            'model__reg_alpha': [0, 0.05, 0.1],
            'model__reg_lambda': [1, 2, 3],
            'model__subsample': [0.3, 0.6, 0.9],
            'model__random_state': [42]
        },
        'scaling': 'unscaled'
    },
    {
        'name': 'LGBM without scaling',
        'model': LGBMRegressor(objective='regression'),
        'params': {
            'model__num_leaves': [30, 50, 70],
            'model__max_depth': [3, 5, 8],
            'model__learning_rate': [0.01, 0.05, 0.1],
            'model__n_estimators': [200, 400, 600],
            'model__subsample': [0.7, 0.8, 0.9],
            'model__colsample_bytree': [0.7, 0.8, 1.0],
            'model__reg_alpha': [0, 0.1, 0.3],
            'model__reg_lambda': [0, 1, 3],
            'model__random_state': [42]
        },
        'scaling': 'unscaled'
    }
]

trainer = ModelTrainer(
    test_set_date=TEST_SET_DATE,
    cut_off_date=CUT_OFF_DATE,
    target_groups=TARGET_GROUPS,
    numerical_cols=NUMERICAL_COLS,
    categorical_cols=CATEGORICAL_COLS,
    boolean_cols=BOOLEAN_COLS,
    target=TARGET,
    model_params=MODEL_PARAMS
)

trainer.grid_search_models()
trainer.extract_shap_values()
trainer.visualize_shap_values()